In [ ]:
#Query Alerce LSST for AGN with probability >0.7 and dec >-30
from alerce.core import Alerce
import pandas as pd
import os

alerce = Alerce()

classifier_name = "stamp_classifier_rubin_beta_20260421"  # double-check this is the live classifier name
target_class    = "AGN"
prob_threshold  = 0.7
dec_min         = -30.0   # degrees — keep only Dec >= -30
page            = 1
page_size       = 500
out_path        = "lsst_agn_fromalerce.csv"

# Start fresh each run (remove old file if present)
if os.path.exists(out_path):
    os.remove(out_path)

header_written = False
total_saved = 0

print("Querying LSST objects...")
while True:
    print(f"Fetching page {page}...")
    try:
        obj_page = alerce.query_objects(
            survey="lsst",
            classifier=classifier_name,   # push probability filtering server-side
            class_name=target_class,
            probability=prob_threshold,
            page=page,
            page_size=page_size,
            format="pandas"
        )
    except Exception as e:
        print("Error fetching page:", e)
        break

    if obj_page is None or len(obj_page) == 0:
        print("No more pages.")
        break

    print(f"Page {page} returned {len(obj_page)} objects (already filtered for {target_class} >= {prob_threshold})")

    # Apply the Dec filter locally
    dec_col = "meandec" if "meandec" in obj_page.columns else \
              "dec"     if "dec"     in obj_page.columns else None

    if dec_col is None:
        print(f"  Warning: no dec/meandec column on page {page} — dec filter skipped for this page.")
        agn_page = obj_page
    else:
        before = len(obj_page)
        agn_page = obj_page[obj_page[dec_col] >= dec_min]
        print(f"  Dec >= {dec_min}: kept {len(agn_page)}/{before}")

    # Save immediately
    if len(agn_page) > 0:
        agn_page.to_csv(out_path, mode="a", index=False, header=not header_written)
        header_written = True
        total_saved += len(agn_page)

    print(f"  Saved this page. Running total: {total_saved}")
    page += 1

print(f"\nDone. Saved {total_saved} AGN candidates to {out_path}")

In [ ]:
#query the list of LSST AGN for ZTF and save to CSV
from alerce.core import Alerce
import pandas as pd
import time
import threading
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm import tqdm
# CONFIG
csv_file = "lsst_agn_fromalerce.csv"
output_csv = "Alerce_LSST_and_ZTF.csv"
search_radius_arcsec = 1.0
MAX_WORKERS = 30        # SPEEDUP: raised from 10. Tune based on rate_limit_hits below —
                         # if those climb fast, dial this back down.
SAVE_EVERY = 200
RETRY_ATTEMPTS = 3
RETRY_DELAY = 2
RESUME = True
REQUEST_TIMEOUT = 20    # seconds before a hung request is abandoned
STALL_TIMEOUT = 120     # if no progress for this many seconds, warn loudly

# ===============================
# THREAD-LOCAL CLIENTS
# ===============================
_thread_local = threading.local()

def get_client():
    if not hasattr(_thread_local, "client"):
        _thread_local.client = Alerce()
        if hasattr(_thread_local.client, "session") and isinstance(_thread_local.client.session, requests.Session):
            session = _thread_local.client.session

            # SPEEDUP / FIX: requests.Session defaults to a connection pool of
            # only 10 connections per host (pool_maxsize=10). With MAX_WORKERS
            # threads all hitting the same API host, anything beyond 10
            # concurrent requests just blocks waiting for a free connection —
            # so raising MAX_WORKERS alone does nothing past 10 without this.
            # Mount an adapter sized to MAX_WORKERS so the pool can't bottleneck you.
            adapter = requests.adapters.HTTPAdapter(
                pool_connections=MAX_WORKERS,
                pool_maxsize=MAX_WORKERS,
            )
            session.mount("https://", adapter)
            session.mount("http://", adapter)

            # patch the underlying requests session with a timeout
            # so no single HTTP call can hang forever
            original_get = session.get
            original_post = session.post
            session.get  = lambda *a, **kw: original_get(*a, timeout=kw.pop("timeout", REQUEST_TIMEOUT), **kw)
            session.post = lambda *a, **kw: original_post(*a, timeout=kw.pop("timeout", REQUEST_TIMEOUT), **kw)
    return _thread_local.client

# ===============================
# SHARED STATE
# ===============================
lock = Lock()
matched_rows = []
checkpoint_counter = 0
rate_limit_hits = 0
last_progress_time = time.time()   # stall detector

# ===============================
# LOAD COORDINATES
# ===============================
df = pd.read_csv(csv_file)
if "meanra" not in df.columns or "meandec" not in df.columns:
    raise ValueError("CSV must contain 'meanra' and 'meandec' columns")

df = df[df['meandec'] > -30].copy()
print(f"{len(df)} objects in ZTF footprint (dec > -30°)")

# ===============================
# RESUME
# ===============================
if RESUME:
    try:
        done_df = pd.read_csv(output_csv)
        done_keys = set(zip(done_df["meanra"].round(6), done_df["meandec"].round(6)))
        df["_key"] = list(zip(df["meanra"].round(6), df["meandec"].round(6)))
        df = df[~df["_key"].isin(done_keys)].drop(columns=["_key"])
        matched_rows = done_df.to_dict("records")
        print(f"Resuming: {len(done_df)} done, {len(df)} remaining.\n")
    except FileNotFoundError:
        print(f"Loaded {len(df)} coordinates. Starting fresh.\n")
else:
    print(f"Loaded {len(df)} coordinates. Starting with {MAX_WORKERS} workers...\n")

# ===============================
# QUERY FUNCTION
# ===============================
def query_row(args):
    index, row = args
    ra = row["meanra"]
    dec = row["meandec"]
    alerce = get_client()

    for attempt in range(RETRY_ATTEMPTS):
        try:
            objs = alerce.query_objects(
                survey="ztf",
                ra=ra,
                dec=dec,
                radius=search_radius_arcsec
            )
            if objs is not None and not objs.empty:
                oids = objs["oid"].tolist()
                row_data = row.to_dict()
                row_data["matched_oids"] = ",".join(oids)
                return row_data, ra, dec, oids
            else:
                return None, ra, dec, []

        except requests.exceptions.Timeout:
            # timeouts are caught explicitly and not retried the same
            # way as other errors — they mean the server is slow/down, not that
            # we did something wrong
            tqdm.write(f"⏱  Timeout at RA={ra:.4f}, DEC={dec:.4f} (attempt {attempt+1})")
            if attempt < RETRY_ATTEMPTS - 1:
                time.sleep(RETRY_DELAY * (attempt + 1))
            else:
                return "timeout", ra, dec, None

        except Exception as e:
            err = str(e).lower()
            is_rate_limit = "429" in err or "rate limit" in err or "too many" in err
            if is_rate_limit:
                time.sleep(RETRY_DELAY * 5 * (attempt + 1))
                return "rate_limit", ra, dec, None
            elif attempt < RETRY_ATTEMPTS - 1:
                time.sleep(RETRY_DELAY * (attempt + 1))
            else:
                return "error", ra, dec, None

# ===============================
# STALL DETECTOR (background thread)
# ===============================
stall_warned = False

def stall_detector():
    global stall_warned
    while True:
        time.sleep(30)
        elapsed = time.time() - last_progress_time
        if elapsed > STALL_TIMEOUT:
            if not stall_warned:
                print(f"\n🚨 STALL DETECTED: No futures completed in {elapsed:.0f}s.")
                print(f"   All {MAX_WORKERS} workers are likely hung on requests.")
                print(f"   This usually means the API is unreachable or very slow.")
                print(f"   Consider: lowering MAX_WORKERS, checking API status, or restarting.")
                stall_warned = True
        else:
            stall_warned = False

detector = threading.Thread(target=stall_detector, daemon=True)
detector.start()

# ===============================
# PARALLEL EXECUTION
# ===============================
timeout_count = 0
error_count = 0
rate_limit_hits = 0
checkpoint_counter = 0
matched_rows = matched_rows or []  # in case RESUME populated it

def save_checkpoint(rows, path):
    if rows:
        pd.DataFrame(rows).to_csv(path, index=False)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(query_row, (i, row)): i for i, row in df.iterrows()}

    with tqdm(total=len(futures), desc="Querying ALeRCE") as pbar:
        for future in as_completed(futures):
            try:
                res, ra, dec, oids = future.result()  # unpack safely
            except Exception as e:
                print("Error with a future:", e)
                res = ra = dec = oids = None

            with lock:
                last_progress_time = time.time()  # reset stall detector

                if res == "rate_limit":
                    rate_limit_hits += 1
                    if rate_limit_hits % 10 == 1:
                        tqdm.write(f"⚠️  Rate limit ({rate_limit_hits}x) — lower MAX_WORKERS if frequent")
                elif res == "timeout":
                    timeout_count += 1
                    if timeout_count % 20 == 1:
                        tqdm.write(f"⏱  Timeouts so far: {timeout_count} — API may be throttling or unstable")
                elif res == "error":
                    error_count += 1
                elif res is not None:
                    matched_rows.append(res)
                    tqdm.write(f"✅ RA={ra:.5f}, DEC={dec:.5f} -> {oids}")

                checkpoint_counter += 1
                if checkpoint_counter % SAVE_EVERY == 0:
                    save_checkpoint(matched_rows, output_csv)
                    tqdm.write(f"Checkpoint saved ({len(matched_rows)} matches so far)")

            pbar.update(1)
# ===============================
# FINAL SAVE + SUMMARY
# ===============================
if matched_rows:
    matched_df = pd.DataFrame(matched_rows)
    matched_df.to_csv(output_csv, index=False)
    print(f"\n✅ Done! {len(matched_rows)} matches saved to '{output_csv}'")
else:
    print("\nNo matches found.")

print(f"\nSummary:")
print(f"  Matches:      {len(matched_rows)}")
print(f"  Rate limits:  {rate_limit_hits}")
print(f"  Timeouts:     {timeout_count}")
print(f"  Other errors: {error_count}")

77068 objects in ZTF footprint (dec > -30°)
Resuming: 779 done, 76289 remaining.



Querying ALeRCE:   0%|          | 199/76289 [00:05<31:03, 40.83it/s] 

Checkpoint saved (779 matches so far)


Querying ALeRCE:   1%|          | 420/76289 [00:10<21:54, 57.71it/s]

Checkpoint saved (779 matches so far)


...

Checkpoint saved (779 matches so far)


Querying ALeRCE:  22%|██▏       | 16403/76289 [53:47<2:17:14,  7.27it/s]

Checkpoint saved (779 matches so far)


Querying ALeRCE:  22%|██▏       | 16517/76289 [54:12<3:16:11,  5.08it/s]



🚨 STALL DETECTED: No futures completed in 123s.
   All 30 workers are likely hung on requests.
   This usually means the API is unreachable or very slow.
   Consider: lowering MAX_WORKERS, checking API status, or restarting.
